# Eksploracja danych · Laboratorium 8
## Regresja: ile wyda klient, który zostaje?

**Sytuacja.** W laboratorium 7 drzewo wskazywało, kto odejdzie, a o kuponie decydowaliśmy przy założeniu, że każdy zatrzymany klient to **250 zł** marży. To uproszczenie: klient, który wydaje 300 zł rocznie, jest wart mniej niż ten, który wydaje 6000 zł. Dziś przewidujemy **liczbę**, a nie klasę: ile klient wyda w kolejnym półroczu. To jest **regresja**.

**Dane:** `klienci_wartosc.csv`: ci sami 1206 klienci co w laboratoriach 3 i 7, z nową kolumną:

| Kolumna | Opis |
|---|---|
| `wydatki_6m` | wartość zamówień od 01.10.2026 do 31.03.2027 [zł]; **0** dla klientów, którzy odeszli |
| pozostałe | jak w laboratorium 7 (stan na 30.09.2026), `odszedl` = 1: brak zamówień w tym półroczu |

**Zasada pracy:** luki `___` z komentarzem `UZUPEŁNIJ`. Decyzje i odpowiedzi zapisujemy w komórkach tekstowych. Przed oddaniem: *Kernel → Restart Kernel and Run All Cells*.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

import sklearn
print("scikit-learn", sklearn.__version__)
pd.set_option("display.precision", 3)

## Z1. Rozgrzewka: prosta na kartce

Pięciu klientów: wydatki w ostatnim roku (x) i w kolejnym półroczu (y). Szukamy prostej **y = a + b·x**, dla której suma kwadratów błędów jest najmniejsza (metoda najmniejszych kwadratów). Wzory:

$$b = \frac{\sum (x - \bar x)(y - \bar y)}{\sum (x - \bar x)^2}, \qquad a = \bar y - b \, \bar x$$

Na kartce: policz średnie, odchylenia, b, a, prognozy, błędy (y − prognoza) i **średni błąd bezwzględny** (MAE: średnia z wartości bezwzględnych błędów). Potem sprawdź kodem.

In [ ]:
t = pd.read_csv("pieciu_klientow.csv", sep=";")
t

In [ ]:
x, y = t["wydatki_rok"], t["wydatki_6m"]
dx, dy = x - x.mean(), y - y.mean()
b = (dx * dy).sum() / (dx ** 2).sum()      # UZUPEŁNIJ: potęga w mianowniku
a = y.mean() - b * x.mean()
prognoza = a + b * x
print(f"b = {b:.3f}   a = {a:.1f}")
print("prognozy:", prognoza.tolist())
print("błędy:   ", (y - prognoza).tolist())
print("MAE:", (y - prognoza).abs().mean())

**Pytanie:** co oznacza b w złotówkach? Ile według prostej wyda w półroczu klient, który w roku wydał 1000 zł?

*Odpowiedź:* Z1: b = 0,45 oznacza, że każda złotówka wydana w roku to średnio 45 groszy wydanych w kolejnym półroczu. Dla klienta, który w roku wydał 1000 zł: 60 + 0,45 · 1000 = **510 zł** w półroczu.

## Z2. Dane: kogo modelujemy?

Klient, który odszedł, ma `wydatki_6m = 0`. To inny proces (odejście) niż „ile wyda, jeśli zostanie”, a odejście przewidywaliśmy już w lab. 7. Regresję uczymy więc tylko na klientach, którzy **zostali**.

In [ ]:
df = pd.read_csv("klienci_wartosc.csv", sep=";", decimal=",")
zostali = df[df["odszedl"] == 0].copy()        # UZUPEŁNIJ: klienci, którzy zostali
print("wszystkich:", len(df), " zostało:", len(zostali))
zostali["wydatki_6m"].describe()

In [ ]:
# konta firmowe znane z laboratorium 3: wydatki powyżej 20 000 zł rocznie
zostali[zostali["wydatki_zl"] > 20000][["nr_karty", "liczba_zamowien", "wydatki_zl", "wydatki_6m"]]

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4.5))
ax[0].scatter(zostali["wydatki_zl"], zostali["wydatki_6m"], s=10, alpha=.5)
ax[0].set_xlabel("wydatki w roku do 30.09.2026 [zł]"); ax[0].set_ylabel("wydatki w kolejnym półroczu [zł]"); ax[0].set_title("wszyscy, którzy zostali")
ind = zostali[zostali["wydatki_zl"] <= 20000]
ax[1].scatter(ind["wydatki_zl"], ind["wydatki_6m"], s=10, alpha=.5)
ax[1].set_xlabel("wydatki w roku [zł]"); ax[1].set_title("bez kont firmowych")
plt.tight_layout(); plt.show()
print("korelacja (bez firm):", round(ind["wydatki_zl"].corr(ind["wydatki_6m"]), 3))

**Zapisz:** ilu klientów modelujemy? Czym różnią się konta firmowe? Jak wygląda zależność na wykresie po prawej?

*Odpowiedź:* Z2–Z3: Modelujemy **825** klientów, którzy zostali, a po odrzuceniu 4 kont firmowych (wydatki > 20 000 zł rocznie) **821 klientów indywidualnych**. Konta firmowe mają wydatki o rząd wielkości większe niż reszta i leżą daleko na osi x. Na wykresie po prawej widać wyraźną dodatnią zależność liniową (korelacja **0,797**): im więcej klient wydał w roku, tym więcej wyda w półroczu. Rozrzut punktów rośnie razem z wydatkami.

## Z3. Jedna firma potrafi przekrzywić prostą

Porównaj prostą dopasowaną do wszystkich, którzy zostali, z prostą bez kont firmowych. Spójrz na konto 101106: w roku wydało ponad 52 tys. zł, w półroczu tylko 2140 zł (utracony kontrakt).

In [ ]:
m_wsz = LinearRegression().fit(zostali[["wydatki_zl"]], zostali["wydatki_6m"])
m_ind = LinearRegression().fit(ind[["wydatki_zl"]], ind["wydatki_6m"])
print(f"z firmami:  wydatki_6m = {m_wsz.intercept_:7.1f} + {m_wsz.coef_[0]:.3f} × wydatki_rok")
print(f"bez firm:   wydatki_6m = {m_ind.intercept_:7.1f} + {m_ind.coef_[0]:.3f} × wydatki_rok")

**Decyzja:** dalej modelujemy **klientów indywidualnych** (bez 4 kont firmowych). Konta firmowe obsługuje handlowiec, a 4 obserwacje to za mało, żeby się czegoś o nich nauczyć.

## Z4. Zbiory i punkt odniesienia

Jak w lab. 7: 30% klientów odkładamy na test. Najprostsza prognoza dla każdego: **średnia** wydatków ze zbioru uczącego. Mierzymy trzy błędy:
* **MAE**: średni błąd bezwzględny w zł („mylimy się średnio o … zł”),
* **RMSE**: pierwiastek ze średniego kwadratu błędu; mocniej karze duże pomyłki,
* **R²**: jaka część zmienności wydatków jest wyjaśniona (1 = idealnie, 0 = nie lepiej niż średnia).

In [ ]:
CECHY = ["liczba_zamowien", "wydatki_zl", "sredni_koszyk_zl", "dni_od_ostatniego",
         "udzial_promocji", "udzial_internet", "liczba_kategorii", "zwroty"]
ucz, test = train_test_split(ind, test_size=0.3, random_state=42)
y_test = test["wydatki_6m"]
print("uczący:", len(ucz), " testowy:", len(test))

def ocena(nazwa, prognoza):
    return {"model": nazwa, "MAE_zl": mean_absolute_error(y_test, prognoza),
            "RMSE_zl": mean_squared_error(y_test, prognoza) ** 0.5, "R2": r2_score(y_test, prognoza)}

wyniki = [ocena("średnia ze zbioru uczącego", np.full(len(test), ucz["wydatki_6m"].mean()))]
pd.DataFrame(wyniki)

## Z5. Regresja prosta: wydatki w roku → wydatki w półroczu

In [ ]:
prosta = LinearRegression().fit(ucz[["wydatki_zl"]], ucz["wydatki_6m"])      # UZUPEŁNIJ: jedna cecha
print(f"wydatki_6m = {prosta.intercept_:.1f} + {prosta.coef_[0]:.3f} × wydatki_zl")
p_prosta = prosta.predict(test[["wydatki_zl"]])
wyniki.append(ocena("regresja prosta (wydatki_zl)", p_prosta))
pd.DataFrame(wyniki)

In [ ]:
# wykres błędów: czy model myli się tak samo dla małych i dużych klientów?
plt.figure(figsize=(8, 4))
plt.scatter(p_prosta, y_test - p_prosta, s=12, alpha=.6)
plt.axhline(0, c="grey")
plt.xlabel("prognoza [zł]"); plt.ylabel("błąd: rzeczywistość − prognoza [zł]")
plt.show()

**Pytanie:** jak interpretujesz współczynnik? O ile model jest lepszy od średniej (MAE)? Co widać na wykresie błędów, gdy prognoza rośnie?

*Odpowiedź:* Z5: Współczynnik b = 0,522: każda dodatkowa złotówka wydana w roku zwiększa prognozę wydatków w półroczu o ok. 52 grosze (a = 27,4 zł to wyraz wolny). MAE spadło z 799,6 zł (średnia) do **375,1 zł**, czyli ponad dwukrotnie; R² = 0,681 wobec −0,029. Na wykresie błędów widać „lejek”: przy małych prognozach błędy są małe, a im większa prognoza, tym większy rozrzut błędów w złotówkach. Model myli się więc bardziej na dużych klientach.

## Z6. Regresja wieloraka: wszystkie cechy naraz

Każda cecha dostaje własny współczynnik: o ile złotych zmienia się prognoza, gdy cecha rośnie o 1, **a pozostałe się nie zmieniają**.

In [ ]:
wiel = LinearRegression().fit(ucz[CECHY], ucz["wydatki_6m"])
print(pd.Series(wiel.coef_, index=CECHY).round(3))
wyniki.append(ocena("regresja wieloraka (8 cech)", wiel.predict(test[CECHY])))
pd.DataFrame(wyniki)

In [ ]:
# cechy powiązane ze sobą: wydatki = liczba zamówień × średni koszyk
ind[["liczba_zamowien", "wydatki_zl", "sredni_koszyk_zl"]].corr().round(2)

**Pytanie:** jaki znak ma współczynnik przy `dni_od_ostatniego` i co znaczy w złotówkach? Dlaczego współczynnik przy `wydatki_zl` spadł w porównaniu z regresją prostą? Czy można z niego wnioskować „podnieśmy klientowi wydatki o 1 zł”?

*Odpowiedź:* Z6: Współczynnik przy `dni_od_ostatniego` jest **ujemny** (−1,894): przy pozostałych cechach bez zmian każdy dodatkowy dzień od ostatniego zamówienia obniża prognozę wydatków w półroczu o ok. 1,89 zł (klient, który dawno nic nie kupił, wyda mniej). Współczynnik przy `wydatki_zl` spadł (0,522 → 0,309), bo cechy są ze sobą skorelowane (wydatki = liczba zamówień × średni koszyk, korelacje 0,54 i 0,78): część wpływu, którą prosta przypisywała samym wydatkom, wieloraka rozdziela na liczbę zamówień i koszyk. Z współczynnika **nie** wynika, że „podniesienie klientowi wydatków o 1 zł” podniesie prognozę o 0,31 zł: to zależność statystyczna „przy pozostałych bez zmian”, a przy skorelowanych cechach nie da się zmienić jednej bez drugich. To nie jest związek przyczynowy. MAE spadło tylko do 363,1 zł (R² 0,708).

## Z7. Logarytm: procenty zamiast złotówek

Pieniądze są skośne: kilku dużych klientów, wielu małych, a błąd rośnie z wielkością klienta (Z5). Po zlogarytmowaniu kwot współczynnik czytamy w **procentach**: wzrost wydatków w roku o 1% → wzrost prognozy o b%. Prognozę wracamy do złotówek funkcją `np.exp`.

In [ ]:
def cechy_log(d):
    return pd.DataFrame({"log_wydatki": np.log(d["wydatki_zl"]), "log_koszyk": np.log(d["sredni_koszyk_zl"]),
                         "dni_od_ostatniego": d["dni_od_ostatniego"]})

logm = LinearRegression().fit(cechy_log(ucz), np.log(ucz["wydatki_6m"]))
print(pd.Series(logm.coef_, index=["log_wydatki", "log_koszyk", "dni_od_ostatniego"]).round(4))
p_log = np.exp(logm.predict(cechy_log(test)))          # UZUPEŁNIJ: z logarytmu z powrotem do złotówek
wyniki.append(ocena("log: wydatki, koszyk, dni", p_log))
pd.DataFrame(wyniki)

**Pytanie:** jak czytasz współczynnik przy `log_wydatki`? A przy `dni_od_ostatniego` (wskazówka: −0,003 na dzień ≈ −0,3% na dzień)? Który model ma najmniejszy MAE?

*Odpowiedź:* Z7: Współczynnik przy `log_wydatki` = 0,516: wzrost wydatków w roku o 1% zwiększa prognozę wydatków w półroczu o ok. 0,52%. Przy `dni_od_ostatniego` = −0,0031: każdy dzień to ok. −0,3% prognozy, a 100 dni to exp(−0,31) ≈ 0,74, czyli ok. −26%. Najmniejszy MAE ma **model logarytmiczny: 342,3 zł** (wieloraka 363,1 zł, prosta 375,1 zł, średnia 799,6 zł).

## Z8. A może drzewo? Drzewo regresyjne z lab. 7

Drzewo regresyjne działa jak klasyfikacyjne, tylko w liściu podaje **średnią** wydatków klientów z tego liścia.

In [ ]:
for g in [2, 3, 4, 6]:
    drz = DecisionTreeRegressor(max_depth=g, min_samples_leaf=20, random_state=0).fit(ucz[CECHY], ucz["wydatki_6m"])   # UZUPEŁNIJ: jak w lab. 7
    wyniki.append(ocena(f"drzewo regresyjne, głębokość {g}", drz.predict(test[CECHY])))
tabela = pd.DataFrame(wyniki).round({"MAE_zl": 1, "RMSE_zl": 1, "R2": 3})
tabela

**Pytanie:** dlaczego drzewo przegrywa tu z regresją liniową? (Wskazówka: ile różnych prognoz daje drzewo o głębokości 3, a ile prosta?)

*Odpowiedź:* Z8: Drzewo o głębokości 3 ma najwyżej 8 liści, więc daje najwyżej 8 różnych prognoz („schodki”), a zależność wydatków jest tu ciągła i w przybliżeniu liniowa: im więcej wydał klient, tym więcej wyda. Prosta (czy model logarytmiczny) daje dla każdego klienta inną prognozę i odtwarza tę zależność gładko, a drzewo musi ją przybliżać stałymi na odcinkach i ma przy tym mało danych w liściach dużych klientów. Dlatego drzewo głębokości 3 ma MAE 398,7 zł wobec 342,3 zł modelu logarytmicznego. Drzewo wygrywa tam, gdzie zależność jest progowa (jak 175 dni w lab. 7), a nie liniowa. Nie ma najlepszego algorytmu: porównujemy modele na tym samym zbiorze testowym.

## Z9. Decyzja o kuponie z wartością klienta

Łączymy dwa modele: **ryzyko** odejścia z drzewa z lab. 7 i **wartość** klienta z modelu logarytmicznego (Z7). Marża to 25% wydatków, a zatrzymany klient zostaje z nami co najmniej rok, czyli dwa półrocza. Oczekiwany zysk z kuponu:

**zysk = ryzyko × 0,30 × 0,25 × 2 × prognozowane wydatki w półroczu − 30 zł**

Porównujemy z regułą z lab. 7 (kupon, gdy ryzyko ≥ 0,4: założenie, że każdy zatrzymany klient jest wart 250 zł).

In [ ]:
# model ryzyka z lab. 7 (te same ustawienia)
CECHY7 = CECHY
Xu, Xt, yu, yt = train_test_split(df[CECHY7], df["odszedl"], test_size=0.3, stratify=df["odszedl"], random_state=42)
drzewo7 = DecisionTreeClassifier(max_depth=3, min_samples_leaf=20, random_state=0).fit(Xu, yu)

klienci = df[df["wydatki_zl"] <= 20000].copy()                     # klienci indywidualni
klienci["ryzyko"] = drzewo7.predict_proba(klienci[CECHY7])[:, 1]
klienci["wartosc_6m"] = np.exp(logm.predict(cechy_log(klienci)))   # ile wyda, jeśli zostanie
klienci["zysk_kuponu"] = klienci["ryzyko"] * 0.30 * 0.25 * 2 * klienci["wartosc_6m"] - 30   # UZUPEŁNIJ: marża 25%

stara = klienci["ryzyko"] >= 0.4
nowa = klienci["zysk_kuponu"] > 0
print("kuponów, reguła z lab. 7:", stara.sum(), "  nowa reguła:", nowa.sum())
print("w obu:", (stara & nowa).sum(), " tylko w starej:", (stara & ~nowa).sum(), " tylko w nowej:", (~stara & nowa).sum())

In [ ]:
cols = ["nr_karty", "wydatki_zl", "dni_od_ostatniego", "ryzyko", "wartosc_6m", "zysk_kuponu"]
print("Przykłady: kupon w starej regule, nie w nowej (wysokie ryzyko, mała wartość)")
print(klienci[stara & ~nowa][cols].sort_values("zysk_kuponu").head(4).round(2).to_string(index=False))
print()
print("Przykłady: kupon tylko w nowej regule (umiarkowane ryzyko, duża wartość)")
print(klienci[~stara & nowa][cols].sort_values("zysk_kuponu", ascending=False).head(4).round(2).to_string(index=False))
print()
print(f"oczekiwany zysk (wg modeli): stara reguła {klienci.loc[stara, 'zysk_kuponu'].sum():.0f} zł, nowa {klienci.loc[nowa, 'zysk_kuponu'].sum():.0f} zł")
print(f"średnia roczna marża (wg modelu) klientów z kuponem w starej regule: {(0.25 * 2 * klienci.loc[stara, 'wartosc_6m']).mean():.0f} zł (założenie z lab. 7: 250 zł)")

**Pytanie:** ile kuponów zmienia adresata? Jakich klientów nowa reguła pomija, a jakich dodaje? Ile naprawdę są warci klienci wysokiego ryzyka w porównaniu z założeniem 250 zł z lab. 7?

*Odpowiedź:* Z9: Reguły różnią się w **349** przypadkach: 125 klientów dostaje kupon w obu regułach, 304 klientów traci kupon (tylko stara), a 45 dostaje go dodatkowo (tylko nowa). Nowa reguła pomija klientów o wysokim ryzyku, ale małej wartości (np. wydatki ok. 60–110 zł rocznie, prognoza 45–105 zł w półroczu), dla których oczekiwany zysk jest ujemny po odjęciu 30 zł kuponu. Dodaje klientów o umiarkowanym ryzyku (ok. 0,22), ale dużej wartości (wydatki 5–10 tys. zł rocznie). Klienci wysokiego ryzyka są w rzeczywistości warci średnio ok. **133 zł** rocznej marży, a nie 250 zł z lab. 7. Oczekiwany zysk (wg modeli): stara reguła −2017 zł, nowa +2326 zł; kupon według samego ryzyka przynosi więc stratę.

## Zadanie domowe
**D1.** Zapisz model z Z7 jako zdanie dla działu marketingu: „klient, który wydał X zł, z koszykiem Y zł i ostatnim zamówieniem Z dni temu, wyda w półroczu ok. … zł”. Policz dla: 1500 zł, 150 zł, 30 dni oraz 1500 zł, 150 zł, 200 dni.

**D2.** W Z9 zmień skuteczność kuponu na 15%. Ilu klientów dostaje kupon według nowej reguły?

**D3.** (dla chętnych) Dodaj do `app.py` z laboratorium 7 pole „prognozowane wydatki w półroczu” i „oczekiwany zysk z kuponu”. Wzór na prognozę: współczynniki modelu z Z7 (wypisz `logm.intercept_` i `logm.coef_`). Wdróż na Render.

In [ ]:
# D1: model z Z7 jako funkcja
def prognoza_6m(wydatki, koszyk, dni):
    x = pd.DataFrame({"log_wydatki": [np.log(wydatki)], "log_koszyk": [np.log(koszyk)], "dni_od_ostatniego": [dni]})
    return float(np.exp(logm.predict(x))[0])

print(logm.intercept_, logm.coef_)
print("30 dni: ", round(prognoza_6m(1500, 150, 30)), "zł")
print("200 dni:", round(prognoza_6m(1500, 150, 200)), "zł")

**D1.** Klient, który wydał X zł, z koszykiem Y zł i ostatnim zamówieniem Z dni temu, wyda w półroczu ok. exp(0,42) · X^0,516 · Y^0,488 · exp(−0,0031 · Z) zł. Dla 1500 zł, koszyka 150 zł: po **30 dniach ok. 700 zł**, po **200 dniach ok. 416 zł** (około 41% mniej).

In [ ]:
# D2: skuteczność kuponu 15%
zysk15 = klienci["ryzyko"] * 0.15 * 0.25 * 2 * klienci["wartosc_6m"] - 30
print("kuponów przy skuteczności 15%:", (zysk15 > 0).sum())

**D2.** Przy skuteczności kuponu 15% według nowej reguły kupon dostaje **19** klientów (zamiast 170 przy 30%): kupon opłaca się tylko przy bardzo dużej wartości klienta i wysokim ryzyku.